# President Tweets on Risk Metrics

James 

# Imports

In [43]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import warnings
warnings.filterwarnings('ignore')

# Set plotting style if needed
import matplotlib.pyplot as plt
plt.style.use('ggplot')

## Load data

### Financial Data

In [44]:
# Load financial datasets
vix = pd.read_csv('data/VIX_DAILY.csv', na_values='.')
treasury = pd.read_csv('data/3mo_treasury.csv', na_values='.')
oil = pd.read_csv('data/oil_px.csv')

# Standardize date columns to datetime
vix['date'] = pd.to_datetime(vix['observation_date'])
treasury['date'] = pd.to_datetime(treasury['observation_date'])
oil['date'] = pd.to_datetime(oil['observation_date'])

# Clean and convert values to numeric
vix['VIXCLS'] = pd.to_numeric(vix['VIXCLS'], errors='coerce')
treasury['DGS3MO'] = pd.to_numeric(treasury['DGS3MO'], errors='coerce')
oil['DCOILWTICO'] = pd.to_numeric(oil['DCOILWTICO'], errors='coerce')

# Merge financial data on date
risk_df = vix[['date', 'VIXCLS']].merge(treasury[['date', 'DGS3MO']].rename(columns={'DGS3MO': 'Treasury_Yield'}), on='date', how='outer')
risk_df = risk_df.merge(oil[['date', 'DCOILWTICO']].rename(columns={'DCOILWTICO': 'Oil_Price'}), on='date', how='outer')

# Sort and calculate daily changes (Risk metric is often the daily change or % change)
risk_df = risk_df.sort_values('date').set_index('date')
risk_df['VIX_Change'] = risk_df['VIXCLS'].diff() # Daily absolute change in VIX
risk_df['Treasury_Return'] = risk_df['Treasury_Yield'].diff()
risk_df['Oil_Ret'] = risk_df['Oil_Price'].pct_change() # Daily return in Oil
risk_df = risk_df.dropna()

In [45]:
risk_df[::-1]

,VIXCLS,Treasury_Yield,Oil_Price,VIX_Change,Treasury_Return,Oil_Ret
date,,,,,,
2022-12-30,21.67,4.42,80.16,0.23,-0.03,0.022058
2022-12-29,21.44,4.45,78.43,-0.70,-0.01,-0.005831
2022-12-28,22.14,4.46,78.89,0.49,0.00,-0.007048
2022-12-23,20.87,4.34,79.57,-1.10,-0.01,0.024331
2022-12-22,21.97,4.35,77.68,1.90,0.02,-0.006268
...,...,...,...,...,...,...
2007-01-10,11.47,5.09,53.95,-0.44,0.01,-0.030548
2007-01-09,11.91,5.08,55.65,-0.09,0.00,-0.007668
2007-01-08,12.00,5.08,56.08,-0.14,0.03,-0.003731


### Twitter Data

In [46]:
# Load Twitter datasets
biden = pd.read_csv('data/JoeBiden.csv')
obama = pd.read_csv('data/obama.csv')
trump = pd.read_csv('data/trump_tweets.csv')

# Standardize text and date columns
biden = biden[['date', 'content']].rename(columns={'content': 'text'})
obama = obama[['Timestamp', 'Text']].rename(columns={'Timestamp': 'date', 'Text': 'text'})
trump = trump[['date', 'text']]

# Add author identifiers
biden['author'] = 'Biden'
obama['author'] = 'Obama'
trump['author'] = 'Trump'

# Combine all tweets
tweets = pd.concat([biden, obama, trump], ignore_index=True)

# 1. Convert to datetime with utc=True to handle mixed formats/zones
tweets['date'] = pd.to_datetime(tweets['date'], format='mixed', utc=True)

# 2. Strip the timezone info (making them 'naive') so they can be easily compared/grouped
tweets['date'] = tweets['date'].dt.tz_localize(None)

# 3. Now extract the date and convert back to datetime for grouping
tweets['date'] = pd.to_datetime(tweets['date'].dt.date)

# 4. Drop any that failed to parse
tweets = tweets.dropna(subset=['date', 'text'])

# Define presidency dates to create the 'is_president' flag
def check_if_president(row):
    d = row['date']
    author = row['author']
    if author == 'Obama' and ('2009-01-20' <= str(d.date()) < '2017-01-20'):
        return 1
    elif author == 'Trump' and ('2017-01-20' <= str(d.date()) < '2021-01-20'):
        return 1
    elif author == 'Biden' and ('2021-01-20' <= str(d.date())):
        return 1
    return 0

tweets['is_president'] = tweets.apply(check_if_president, axis=1)
obama['date'] = pd.to_datetime(obama['date'], errors='coerce')
trump['date'] = pd.to_datetime(trump['date'], errors='coerce')

# Group tweets by day (concatenate text, take max of is_president)
daily_tweets = tweets.groupby(['date']).agg({
    'text': lambda x: ' '.join(x.astype(str)),
    'is_president': 'max',
    'author': lambda x: ', '.join(x.unique()) # Lists unique authors for that day
}).reset_index()

### Merge dfs

In [47]:
# Merge the daily tweets with the daily risk metrics
df_merged = risk_df.merge(daily_tweets.set_index('date'), left_index=True, right_index=True, how='inner')

# Clean the combined text: lowercasing, removing URLs, special characters
def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation/numbers
    return text.lower()

df_merged['clean_text'] = df_merged['text'].apply(clean_text)

## NLP Text extraction

In [48]:
# Using TF-IDF to extract top 100 most frequent/important keywords to prevent overfitting
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_merged['clean_text'])

# Create a DataFrame for the TF-IDF features
keyword_features = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=vectorizer.get_feature_names_out(),
    index=df_merged.index
)

# Join the keyword features back to our main dataset
df_final = pd.concat([df_merged, keyword_features], axis=1)
df_final = df_final.dropna()

### regression

In [49]:
[x for x in df_final.columns]

['VIXCLS',
 'Treasury_Yield',
 'Oil_Price',
 'VIX_Change',
 'Treasury_Return',
 'Oil_Ret',
 'text',
 'is_president',
 'author',
 'clean_text',
 'america',
 'american',
 'americans',
 'amp',
 'apr',
 'aug',
 'bad',
 'barack',
 'barackobama',
 'best',
 'better',
 'biden',
 'big',
 'border',
 'campaign',
 'care',
 'china',
 'country',
 'day',
 'deal',
 'dec',
 'democrats',
 'did',
 'doing',
 'donald',
 'dont',
 'economy',
 'election',
 'fake',
 'feb',
 'foxandfriends',
 'foxnews',
 'going',
 'good',
 'got',
 'great',
 'hard',
 'help',
 'hillary',
 'history',
 'house',
 'im',
 'jan',
 'job',
 'jobs',
 'joe',
 'jul',
 'jun',
 'just',
 'know',
 'like',
 'look',
 'love',
 'make',
 'mar',
 'media',
 'nation',
 'national',
 'need',
 'new',
 'news',
 'night',
 'nov',
 'obama',
 'oct',
 'people',
 'pm',
 'president',
 'realdonaldtrump',
 'really',
 'republican',
 'republicans',
 'right',
 'rt',
 'run',
 'said',
 'sep',
 'state',
 'states',
 'support',
 'thank',
 'thanks',
 'think',
 'time',
 'tod

In [50]:
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
# Fit Ordinary Least Squares (OLS) Regression
# model = sm.OLS(Y, X).fit()
model = smf.ols(formula="Oil_Ret ~ is_president + C(author)", data=df_final).fit()

# Print the regression summary
# print(model.summary2())

sg = Stargazer([model])
sg.title("OLS with Presidential Control Results")

with open("outfile.html", "w") as f:
    f.write(sg.render_html())


## Final words 

In [51]:
# regression for keywords
Y = df_final['Oil_Ret'] 

# Predictors: Keyword frequencies + the "is_president" effect + a constant intercept
X = df_final[['is_president'] + list(keyword_features.columns)]
X = sm.add_constant(X)

# Fit Ordinary Least Squares (OLS) Regression
model = sm.OLS(Y, X).fit()

# Extract P-values and Coefficients from the model
results_df = pd.DataFrame({
    'Coefficient': model.params,
    'P-Value': model.pvalues
})

# Filter out the constant to focus purely on features
results_df = results_df.drop('const', errors='ignore')

# Determine significance (p < 0.05) and sort by absolute coefficient impact
significant_factors = results_df[results_df['P-Value'] < 0.1]
significant_factors['Abs_Impact'] = significant_factors['Coefficient'].abs()
significant_factors = significant_factors.sort_values(by='Abs_Impact', ascending=False)

print("Top Impactful Words / Features (Statistically Significant):")
display(significant_factors.head(20))

# Specifically check the effect of being in office
print("\nEffect of being President on Risk:")
if 'is_president' in significant_factors.index:
    effect = significant_factors.loc['is_president', 'Coefficient']
    print(f"Statistically significant effect. Coefficient: {effect:.4f}")
else:
    print(f"Not statistically significant at p < 0.05. Coefficient was {results_df.loc['is_president', 'Coefficient']:.4f} (p-value: {results_df.loc['is_president', 'P-Value']:.4f})")

Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
states,-0.085979,0.001453,0.085979
united,0.082353,0.002299,0.082353
doing,-0.065734,0.002952,0.065734
joe,0.048647,0.088658,0.048647
got,-0.046821,0.067564,0.046821
look,-0.045873,0.053619,0.045873
nation,-0.041234,0.023203,0.041234
pm,-0.040774,0.008457,0.040774
state,0.037268,0.071336,0.037268



Effect of being President on Risk:
Not statistically significant at p < 0.05. Coefficient was 0.0009 (p-value: 0.8190)


In [52]:
new_sig = significant_factors.copy()
for c in new_sig.columns:
    new_sig[c] = new_sig[c].apply(lambda x: round(x, 3))

with open("outfile.html", "w") as f:
    f.write(new_sig.to_html())

# regression 2: TEB

In [55]:
import statsmodels.formula.api as smf

for word in significant_factors.reset_index()['index']:
    df_final[word] = df_final['text'].apply(lambda x: word in x)


# Fit Ordinary Least Squares (OLS) Regression
# model = sm.OLS(Y, X).fit()
model1 = smf.ols(formula=f"Oil_Ret ~ is_president + C(author)", data=df_final).fit()
model2 = smf.ols(formula=f"Oil_Ret ~ is_president + {' + '.join(word for word in significant_factors.reset_index()['index'])}", data=df_final).fit()
model3 = smf.ols(formula=f"Oil_Ret ~ is_president + C(author) + {' + '.join(word for word in significant_factors.reset_index()['index'])}", data=df_final).fit()
model3 = smf.ols(formula=f"Oil_Ret ~ is_president + {' + '.join(f'C(author)*{word}' for word in significant_factors.reset_index()['index'])}", data=df_final).fit()

sg = Stargazer([model1, model2, model3])
sg.title("OLS with Presidential and Keyword controls")

with open("outfile.html", "w") as f:
    f.write(sg.render_html())